In [1]:
import os

REPO_URL = "https://github.com/sinh2206/O_D.git"
REPO_DIR = "O_D"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch --all --prune
    !git checkout -B $(git symbolic-ref --short refs/remotes/origin/HEAD | sed 's@^origin/@@')
    !git reset --hard origin/$(git symbolic-ref --short refs/remotes/origin/HEAD | sed 's@^origin/@@')
    %cd ..

%cd {REPO_DIR}
!git log -1 --oneline


Cloning into 'O_D'...
remote: Enumerating objects: 9373, done.
remote: Counting objects: 100% (342/342), done.
remote: Compressing objects: 100% (297/297), done.
remote: Total 9373 (delta 117), reused 266 (delta 44), pack-reused 9031 (from 2)
Receiving objects: 100% (9373/9373), 1009.72 MiB | 44.46 MiB/s, done.
Resolving deltas: 100% (118/118), done.
Updating files: 100% (9022/9022), done.
/content/O_D
21d6440 (HEAD -> main, origin/main, origin/HEAD) ver1.8


In [2]:
!pwd
!ls


/content/O_D
LICENSE  object_detection.ipynb  public     requirements.txt  train.py
models	 predict.py		 README.md  results	      utils


In [3]:
%pip install -r requirements.txt

In [9]:
!python train.py \
  --train_data ./public/annotations/train.json \
  --val_data ./public/annotations/val.json \
  --image_dir ./public/train/images \
  --val_image_dir ./public/val/images \
  --checkpoint_dir ./models/

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/content/O_D/train.py:523: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
Device: cuda, AMP: True
Train samples: 7500, Val samples: 1500, Classes: ['person', 'car', 'dog', 'cat', 'chair']
Balanced sampling: True
Class weights enabled: True
Class weights: [0.5563, 1.1696, 1.2632, 1.3591, 0.6519]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning

In [10]:
!python predict.py \
  --image_dir ./public/val/images \
  --output val_predictions.json \
  --model_path ./models/best.pth

Device: cuda
Predicted images: 1500
Saved JSON: val_predictions.json
Saved preview images: 50 -> results


In [11]:
!python public/tools/evaluate_predictions.py \
  --ground_truth public/annotations/val.json \
  --predictions val_predictions.json \
  --output val_score.json

{
  "mAP@0.5": 0.470397,
  "performance_points": 15,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 2021,
  "num_predictions": 3926,
  "micro_precision": 0.298268,
  "micro_recall": 0.579416,
  "per_class": {
    "person": {
      "ap": 0.572611,
      "num_ground_truth": 1074,
      "num_predictions": 2512,
      "true_positives": 731,
      "false_positives": 1781,
      "recall": 0.680633,
      "precision": 0.291003
    },
    "car": {
      "ap": 0.432857,
      "num_ground_truth": 283,
      "num_predictions": 630,
      "true_positives": 159,
      "false_positives": 471,
      "recall": 0.561837,
      "precision": 0.252381
    },
    "dog": {
      "ap": 0.63037,
      "num_ground_truth": 206,
      "num_predictions": 527,
      "true_positives": 148,
      "false_positives": 379,
      "recall": 0.718447,
      "precision": 0.280835
    },
    "cat": {
      "ap": 0.712602,
      "num_ground_truth": 176,
      "num_predictions": 256,
      "true_positives": 132,
      "f

In [12]:
# Zip project excluding public
import os
from pathlib import Path

src = Path('/content/O_D')
out_zip = Path('/content/train.zip')

if not src.exists():
    raise FileNotFoundError(f'Not found: {src}')

cmd = f"cd {src} && zip -r {out_zip} . -x 'public/*' 'public/**'"
print(cmd)
ret = os.system(cmd)
if ret != 0:
    raise RuntimeError(f'zip failed with code {ret}')
print(f'Created: {out_zip}')


cd /content/O_D && zip -r /content/train.zip . -x 'public/*' 'public/**'
Created: /content/train.zip


In [8]:
from google.colab import files
files.download('/content/train.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>